# Tour of the Model Zoo

`pyfieldml.datasets` ships ten FieldML documents: five synthetic/reference assets and five real BodyParts3D (BP3D) anatomical meshes. This notebook renders the whole gallery, assembles a multi-part MSK scene, and summarizes the dataset sizes.

Each dataset carries its upstream license, citation, and origin URL; nothing here has hidden provenance.

In [ ]:
import numpy as np
import pyvista as pv

from pyfieldml import datasets
from pyfieldml.interop.pyvista import to_pyvista

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

## The gallery

Ten datasets, ten panels. Color encodes dataset family:
- **blue** = synthetic / reference (`unit_cube`, `femur` CSG, `rectus_femoris`, `bunny_stanford`)
- **orange** = BodyParts3D real anatomy (`femur_bodyparts3d`, `vertebra_l3`, `scapula`, `tibia_left`, `hip_bone_left`, `skull`)

In [ ]:
GALLERY = [
    ("unit_cube", "synthetic", "smoke-test hex primitive"),
    ("femur", "synthetic", "CSG synthetic + BMD field"),
    ("rectus_femoris", "synthetic", "fiber-direction vector field"),
    ("bunny_stanford", "reference", "public-domain benchmark"),
    ("femur_bodyparts3d", "BP3D", "atlas-real anatomy"),
    ("vertebra_l3", "BP3D", "foramen topology (spine SSM)"),
    ("scapula", "BP3D", "thin curved shell"),
    ("tibia_left", "BP3D", "bilateral companion to femur"),
    ("hip_bone_left", "BP3D", "pelvic MSK pivot"),
    ("skull", "BP3D", "74-component messy real data"),
]
COLORS = {
    "synthetic": "skyblue",
    "reference": "wheat",
    "BP3D": "peachpuff",
}

docs = {n: datasets.load(n) for n, _, _ in GALLERY}
grids = {n: to_pyvista(d) for n, d in docs.items()}
for name, _, _ in GALLERY:
    g = grids[name]
    print(f"  {name:22s}  n_points={g.n_points:6d}  n_cells={g.n_cells:6d}")

In [ ]:
p = pv.Plotter(off_screen=True, window_size=(1500, 700), shape=(2, 5))
for i, (name, family, note) in enumerate(GALLERY):
    p.subplot(i // 5, i % 5)
    g = grids[name]
    p.add_mesh(g, color=COLORS[family], show_edges=False)
    p.add_text(
        f"{name}\n{g.n_points} nodes / {g.n_cells} cells",
        font_size=8,
        color="black",
        position="upper_edge",
    )
    p.add_text(note, font_size=7, color="dimgray", position="lower_edge")
    p.view_isometric()
p.show(screenshot="/tmp/zoo_gallery.png")

## Vertebra-L3 close-up: wireframe + edge-colored foramen

The lumbar-3 vertebra is the gateway dataset for spine SSM work: it has a **foramen** (the central hole through which the spinal cord runs), which is non-trivial topology for statistical shape models. The wireframe panel reveals triangulation density; the edge-colored panel makes the foramen boundary visually obvious.

In [ ]:
vert = grids["vertebra_l3"]
p = pv.Plotter(off_screen=True, window_size=(900, 450), shape=(1, 2))
p.subplot(0, 0)
p.add_mesh(vert, style="wireframe", color="steelblue", line_width=1)
p.add_text("L3 vertebra - wireframe", font_size=10)
p.view_vector((1, 0, 0.3))
p.subplot(0, 1)
p.add_mesh(
    vert,
    color="lightcoral",
    show_edges=True,
    edge_color="black",
    line_width=1,
    opacity=0.92,
)
p.add_text("L3 vertebra - edge-highlighted foramen", font_size=10)
p.view_vector((1, 0, 0.3))
p.show(screenshot="/tmp/vertebra_closeup.png")

## Clinical MSK assembly

Loading four BP3D parts (left femur, left tibia, left hip bone, L3 vertebra) into the same pyvista scene demonstrates how pyfieldml handles multi-mesh assemblies. The pieces are not surgically co-registered (they come from different BP3D donors), so we roughly stack them along the z-axis using each part's own bounding box - enough to make the MSK layout visually coherent.

In [ ]:
PARTS = ["vertebra_l3", "hip_bone_left", "femur_bodyparts3d", "tibia_left"]
COLOR_MAP = {
    "vertebra_l3": "indianred",
    "hip_bone_left": "mediumseagreen",
    "femur_bodyparts3d": "cornflowerblue",
    "tibia_left": "goldenrod",
}


def stacked_copy(g, z_offset):
    gc = g.copy()
    bounds = np.array(gc.bounds).reshape(3, 2)
    center_xy = bounds[:2].mean(axis=1)
    z_min = bounds[2, 0]
    gc.translate((-center_xy[0], -center_xy[1], -z_min + z_offset), inplace=True)
    return gc


# Rough anatomical stack: vertebra on top, hip below, femur lower, tibia bottom
OFFSETS = {
    "vertebra_l3": 240.0,
    "hip_bone_left": 120.0,
    "femur_bodyparts3d": -180.0,
    "tibia_left": -440.0,
}

p = pv.Plotter(off_screen=True, window_size=(700, 900))
for name in PARTS:
    gc = stacked_copy(grids[name], OFFSETS[name])
    p.add_mesh(gc, color=COLOR_MAP[name], show_edges=False, label=name)
p.add_legend(bcolor=(1, 1, 1), size=(0.2, 0.2))
p.add_text("Left-side MSK assembly (roughly stacked)", font_size=10)
p.view_vector((1, 0.2, 0.3))
p.show_axes()
p.show(screenshot="/tmp/msk_assembly.png")

## Provenance card for every dataset

License, citation, origin - printed inline so this notebook is a stand-alone reference without needing to crack open the registry.

In [ ]:
for name, _, _ in GALLERY:
    info = datasets.info(name)
    print(f"== {name} ==")
    for k, v in info.items():
        print(f"  {k:10s}: {v}")
    print()

## Dataset-size summary

Node count per dataset, on a log scale.

In [ ]:
import matplotlib.pyplot as plt

names = [n for n, _, _ in GALLERY]
nps = [grids[n].n_points for n in names]
ncs = [grids[n].n_cells for n in names]

fig, ax = plt.subplots(figsize=(10, 3.5))
x = np.arange(len(names))
ax.bar(x - 0.2, nps, width=0.4, label="n_points", color="steelblue")
ax.bar(x + 0.2, ncs, width=0.4, label="n_cells", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_yscale("log")
ax.set_ylabel("count (log)")
ax.set_title("Model zoo: nodes and cells per dataset")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

### Why ship real anatomy?

Synthetic primitives are great for unit tests but terrible for showing off spatial resolution, curvature handling, or visual debugging. The bundled real meshes let tutorials, benchmarks, and bug repros work offline with zero configuration while staying license-clean.

The BP3D meshes are distributed under **CC-BY-SA 2.1 JP** (DBCLS); the Stanford Bunny is **public domain**; the synthetic assets are CC0 within the pyfieldml source tree.